# Making it Agentic

Our set-up so far has been using RAG, which, as shown in the previous tutorial, involves passing the vector search results to the model, with the vector search being performed before the LLM sees the prompt or the question. 

In the last year, the popularity of this basic RAG pipeline has declined, with a move towards autonomous, agentic systems. This means, providing the LLM tools and letting them decide what to do with them. A tool based set-up has only been made possible by recent LLMs which are trained to know how to perform tool calls. 

LLMs which can fit onto a laptop are generally not going to be good enough to call tools, unless your laptop is unusually high spec. So, in this section we are going to move to using an OpenAI API key and send out requests to OpenAI so we can use top models without worry about the compute performance. 


Prerequisite: **OpenAI API key** 

You're recommended to save the key in a text file called `.env` with the format `OPENAI_API_KEY=sk-abcdefg1234` this will be loaded below. 

First, install the dotenv package which loads the `.env` file:


In [1]:
# pip install dotenv

In [2]:
from dotenv import load_dotenv
import os 
load_dotenv()

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

if OPENAI_API_KEY: 
    print("API key found!")

API key found!


Great! Now we are again going to use Langchain to create and orchestrate our agents. This time though, we will use OpenAI as our model provider, if you haven't already you'll need to install langchain-openai with the following cell: 

In [3]:
# pip install langchain-openai

In [4]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# Create OpenAI chatbot
llm = ChatOpenAI(model = "gpt-5-nano") 

response = llm.invoke([("human", "Please tell me a joke?")])
print(response.content)

Sure! Why don’t scientists trust atoms? Because they make up everything. 

Want another one?


## Creating tools 

Tools are functions which an AI model can invoke by sending a structured response with input parameters. The tool runs on your computer but the tool definition and usage is sent to the LLM, which can then respond with a request to use a tool. The tool output is then sent back to the model. 

Langchain makes it very easy to package and use tools with their Agent class. So lets say we had a basic calculation function.

In [5]:
def addNumbers(a:float, b:float) -> float:
    "Takes two numbers as input and returns the sum of the numbers"
    return a + b

We can turn this to an agent tool very simply, using the langchain tool decorator. The key thing is we use the 

In [6]:
from langchain.tools import tool

@tool("add_numbers")
def addNumbers(a:float, b:float) -> float:
    "Takes two numbers as input and returns the sum of the numbers"
    return a + b


Its as simple as just adding a decorator! Before binding this to the model, lets just see what this actually looks like when it gets sent to the model: 

In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool

# Prints the raw JSON tool definition. This is what the model sees.
print(convert_to_openai_tool(addNumbers))

{'type': 'function', 'function': {'name': 'add_numbers', 'description': 'Takes two numbers as input and returns the sum of the numbers', 'parameters': {'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}}, 'required': ['a', 'b'], 'type': 'object'}}}


## Bind tools to an agent

Again langchain makes this step very easy, we just use the create_agent() function

In [8]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage
agent = create_agent(llm, tools=[addNumbers]) 


response = agent.invoke({
    "messages": [
        SystemMessage(content="Please use the available tools to answer the question"),
        HumanMessage(content="What is 7830.582 + 4152789.3?")
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ System Message ================================

Please use the available tools to answer the question
================================ Human Message =================================

What is 7830.582 + 4152789.3?
================================== Ai Message ==================================
Tool Calls:
  add_numbers (call_Kl3UKl0f8aVSJmTW1qIwzeq6)
 Call ID: call_Kl3UKl0f8aVSJmTW1qIwzeq6
  Args:
    a: 7830.582
    b: 4152789.3
================================= Tool Message =================================
Name: add_numbers

4160619.8819999998
================================== Ai Message ==================================

4,160,619.882


Great! We can see it called the add_numbers tool in the first Ai Message and recieved the response, which it could then reason upon (format for readability and ignore precision errors) and return to the user.

Now we know how to create tools, lets return back to the main project! 

## Adding vector search as a tool

In the previous notebook, we created a RAG structure where the results of a vector search were automatically piped to the agent. Lets use the same function but use it as a tool instead. 

First lets define the global variables required for the set-up:

In [9]:
import iris
from sentence_transformers import SentenceTransformer

conn = iris.connect("localhost", 32782, "DEMO", "_SYSTEM", "ISCDEMO") # Server, Port , Namespace, Username, Password
cursor = conn.cursor()

table_name = "VectorSearch.DocRefVectors"
model = SentenceTransformer('all-MiniLM-L6-v2') 


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Then lets use the function we defined in the previous section, but this time add a tool decorator.

In [10]:
@tool("medical_history_search", description="Tool to perform a vector search over the medical notes of a particular patient and return the results with the closest match")
def vector_search(prompt:str,patient_id:int):
    search_vector =  model.encode(prompt, normalize_embeddings=True,show_progress_bar=False ).tolist() 
    
    search_sql = f"""
        SELECT TOP 3 ClinicalNotes 
        FROM {table_name}
        WHERE PatientID = ?
        ORDER BY VECTOR_COSINE(NotesVector, TO_VECTOR(?,double)) DESC
    """
    cursor.execute(search_sql,[patient_id, str(search_vector)])
    
    results = cursor.fetchall()
    return results

Now lets define the agent with bound tools, memory and a suitable system prompt: 

In [11]:

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama

# Initialize model
llm = ChatOpenAI(model = "gpt-5-nano") 

# Initialise short-term memory
checkpointer = InMemorySaver()

system_prompt = """You are a medical history chatbot. You're job is to use the medical history vector search tool provided to provide suitable 
                information to the user about the patients medical history and return it in a usable format. You can provide the users prompt 
                directly to the search, or rephrase it for more accurate results. 

                Please only use information returned by the vector search. If no relevant information is availble, please state that no relevant 
                information was returned by the search. 
                """   

# Create model
medical_search_bot_agent = create_agent(
    model=llm, # Set model as our LLM 
    # Creates the agent's memory with pre-initialized model
    checkpointer=checkpointer,
    tools = [vector_search],
    system_prompt = system_prompt
)




In [12]:
config = {"configurable": {"thread_id": "1001"}}

response = medical_search_bot_agent.invoke( {"messages":[
    HumanMessage(content="My patient (id = 3) is presenting with a respiratory complaint. Has she had similar issues in the past?")
                 ]}, config)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

My patient (id = 3) is presenting with a respiratory complaint. Has she had similar issues in the past?
================================== Ai Message ==================================
Tool Calls:
  medical_history_search (call_cmzKW6iq6CHudLlWWKyzTTJ3)
 Call ID: call_cmzKW6iq6CHudLlWWKyzTTJ3
  Args:
    prompt: Past history of respiratory issues: has patient with ID 3 had similar respiratory complaints in the past (e.g., cough, wheeze, pneumonia, asthma, COPD)?
    patient_id: 3
================================= Tool Message =================================
Name: medical_history_search

[["Date: 2024-11-21\nProvider: Dr. Chin306 Kulas532\nLocation: Waltham Urgent Care\nReason for Visit: Cough and difficulty breathing\nSubjective:\nAurora was brought in with a 3-day history of persistent cough, mild fever, and labored breathing. Parent reports decreased appetite and increased fatigue. No vomiting or diar

# Conclusions

In this step, we see how a vector search can be provided to the model as a callable tool, rather than a structured step. It also demonstrates how Langchain can be used to turn any function into an agentic tool. 

We could extend this by building an Model Context Protocol server - this is a method to expose tools to external agents, so the agent harness and vector search could be run on different computers. 

